# Aula 2 · Abertura — Do post-it ao número: fechando o AI Business Model Canvas
**FECAP · Pós em Data Analytics e IA aplicada a Negócios · Módulo 8: Versionamento e Governança de Produtos de IA**

Hoje fechamos a demonstração com o exemplo da Loja Aurora em três passos:

1. **Colocar número nos blocos** que mais pesam na decisão: tamanho da dor (bloco 2), valor gerado (bloco 5) e custos (bloco 9).
2. **Transformar o canvas em código**: um arquivo que gera a figura e pode ser versionado no GitLab, como qualquer outro artefato do projeto.
3. **Descobrir o elo com a aula de hoje**: todo número do canvas depende de uma definição que ainda não fizemos com cuidado, a **target**.

> Execute as células em ordem com **Shift + Enter**. Tempo estimado: 20 minutos.
> Todos os dados são fictícios (base sintética Loja Aurora v1, semente 42).

## 1. Recriar a base da Loja Aurora (a mesma da Aula 1)
A célula abaixo grava o gerador no Colab. Com a **mesma semente**, o mesmo código e as mesmas bibliotecas, todos na sala obtêm a mesma base.

In [0]:
%%writefile loja_aurora_gerador.py
"""
Loja Aurora — gerador da base sintética do Módulo 8 (FECAP)
Versionamento e Governança de Produtos de IA

Gera 5 tabelas usadas nas 7 aulas:
  clientes, produtos, lojas            -> dimensões
  transacoes                           -> Case A: prevenção a fraude (classificação)
  vendas_semanais                      -> Case B: previsão de vendas (regressão)

Uso:
  from loja_aurora_gerador import gerar_base
  tabelas = gerar_base(seed=42)          # base oficial da disciplina (v1)
  tabelas = gerar_base(seed=None)        # base diferente a cada execução (lição de reprodutibilidade)

Requer apenas numpy e pandas (já instalados no Colab e no Databricks).
Todos os dados são fictícios.
"""

import hashlib
import numpy as np
import pandas as pd

VERSAO_GERADOR = "1.0.0"
DATA_INICIO = pd.Timestamp("2024-01-01")
DATA_CORTE = pd.Timestamp("2026-10-31 23:59:59")   # "hoje" dentro da base

UFS = ["SP", "RJ", "MG", "PR", "RS", "BA", "PE", "SC", "GO", "DF"]
PESO_UF = np.array([0.38, 0.13, 0.11, 0.07, 0.06, 0.06, 0.05, 0.05, 0.05, 0.04])

CATEGORIAS = {
    # categoria: (n_produtos, preco_min, preco_max, margem, sazonal_natal)
    "Eletrônicos": (20, 199, 4999, 0.18, 1.9),
    "Casa Inteligente": (12, 89, 1499, 0.30, 1.6),
    "Eletrodomésticos": (16, 149, 3999, 0.22, 1.4),
    "Moda": (24, 39, 399, 0.55, 1.7),
    "Beleza": (18, 19, 249, 0.50, 1.8),
    "Esporte": (14, 49, 1299, 0.40, 1.3),
    "Brinquedos": (16, 29, 599, 0.45, 2.6),
}


def _rng(seed):
    return np.random.default_rng(seed)


def _datas_aleatorias(rng, n, inicio, fim, pesos_mensais=None):
    """Sorteia datas-hora entre inicio e fim, com crescimento e picos sazonais."""
    dias = pd.date_range(inicio.normalize(), fim.normalize(), freq="D")
    t = np.arange(len(dias)) / len(dias)
    peso = 1 + 0.6 * t                                    # crescimento da operação
    mes, dia = dias.month.values, dias.day.values
    peso = peso * np.where(mes == 12, 1.6, 1.0)            # Natal
    bf = np.zeros(len(dias), dtype=bool)
    for ano in (2024, 2025, 2026):                          # semana da Black Friday
        bf_dia = pd.Timestamp(f"{ano}-11-01") + pd.offsets.Week(weekday=4) * 4
        bf |= (dias >= bf_dia - pd.Timedelta(days=3)) & (dias <= bf_dia + pd.Timedelta(days=3))
    peso = peso * np.where(bf, 3.0, 1.0)
    peso = peso * np.where((mes == 5) & (dia <= 12), 1.3, 1.0)   # Dia das Mães
    peso = peso / peso.sum()
    idx = rng.choice(len(dias), size=n, p=peso)
    # hora do dia: pico 10h-22h, cauda de madrugada
    horas = rng.choice(24, size=n, p=_perfil_horas())
    minutos = rng.integers(0, 60, n)
    segundos = rng.integers(0, 60, n)
    return (pd.to_datetime(dias.values[idx]) + pd.to_timedelta(horas, "h")
            + pd.to_timedelta(minutos, "m") + pd.to_timedelta(segundos, "s"))


def _perfil_horas():
    p = np.array([1.2, 0.8, 0.5, 0.4, 0.4, 0.6, 1.2, 2.2, 3.5, 4.8, 5.8, 6.2,
                  6.5, 6.3, 6.0, 5.9, 6.0, 6.4, 6.9, 7.3, 7.6, 7.0, 5.0, 2.9])
    return p / p.sum()


# ----------------------------------------------------------------------------
# Dimensões
# ----------------------------------------------------------------------------
def gerar_lojas():
    return pd.DataFrame({
        "id_loja": ["L01", "L02", "L03", "L04", "L05", "L06", "ECOM"],
        "nome_loja": ["Aurora Paulista", "Aurora Campinas", "Aurora Barra", "Aurora Savassi",
                      "Aurora Curitiba", "Aurora Salvador", "Aurora Digital (site + app)"],
        "cidade": ["São Paulo", "Campinas", "Rio de Janeiro", "Belo Horizonte", "Curitiba",
                   "Salvador", "Nacional"],
        "uf": ["SP", "SP", "RJ", "MG", "PR", "BA", "BR"],
        "tipo": ["física"] * 6 + ["digital"],
        "data_inauguracao": pd.to_datetime(["2015-03-10", "2018-08-22", "2016-11-05", "2019-04-17",
                                            "2021-09-30", "2023-06-12", "2017-01-15"]),
    })


def gerar_produtos(rng):
    linhas, i = [], 1
    for cat, (n, pmin, pmax, margem, _) in CATEGORIAS.items():
        precos = np.exp(rng.uniform(np.log(pmin), np.log(pmax), n)).round(-1) - 0.10
        for p in precos:
            linhas.append({
                "id_produto": f"P{i:04d}",
                "categoria": cat,
                "nome_produto": f"{cat} — item {i:04d}",
                "preco_lista": float(max(p, 9.90)),
                "custo_unitario": float(round(max(p, 9.90) * (1 - margem * rng.uniform(0.8, 1.2)), 2)),
                "data_lancamento": (DATA_INICIO - pd.Timedelta(days=int(rng.integers(0, 900)))
                                    if rng.random() < 0.8 else
                                    DATA_INICIO + pd.Timedelta(days=int(rng.integers(30, 900)))),
            })
            i += 1
    return pd.DataFrame(linhas)


def gerar_clientes(rng, n=30000):
    data_cad = _datas_aleatorias(rng, n, pd.Timestamp("2019-01-01"), DATA_CORTE - pd.Timedelta(days=1)).normalize()
    segmento = rng.choice(["novo", "recorrente", "vip"], size=n, p=[0.45, 0.45, 0.10])
    faixa_renda = rng.choice(["até 3 SM", "3 a 6 SM", "6 a 10 SM", "acima de 10 SM"], size=n,
                             p=[0.35, 0.33, 0.20, 0.12]).astype(object)
    faixa_renda[rng.random(n) < 0.08] = None                 # dado faltante (qualidade de dados)
    return pd.DataFrame({
        "id_cliente": [f"C{i:06d}" for i in range(1, n + 1)],
        "data_cadastro": data_cad,
        "uf": rng.choice(UFS, size=n, p=PESO_UF),
        "faixa_etaria": rng.choice(["18-24", "25-34", "35-44", "45-59", "60+"], size=n,
                                   p=[0.14, 0.31, 0.27, 0.20, 0.08]),
        "canal_aquisicao": rng.choice(["orgânico", "mídia paga", "indicação", "marketplace", "loja física"],
                                      size=n, p=[0.28, 0.30, 0.12, 0.10, 0.20]),
        "segmento": segmento,
        "faixa_renda": faixa_renda,
        "aceita_comunicacao": rng.random(n) < 0.62,
    })


# ----------------------------------------------------------------------------
# Case A — transações (prevenção a fraude)
# ----------------------------------------------------------------------------
def gerar_transacoes(rng, clientes, produtos, n=180000):
    dt = _datas_aleatorias(rng, n, DATA_INICIO, DATA_CORTE)
    dt = pd.Series(dt).sort_values().reset_index(drop=True)

    # clientes: VIP e recorrentes compram mais
    peso_cli = clientes["segmento"].map({"novo": 1.0, "recorrente": 2.2, "vip": 4.0}).values
    idx_cli = rng.choice(len(clientes), size=n, p=peso_cli / peso_cli.sum())
    cli = clientes.iloc[idx_cli].reset_index(drop=True)
    # cliente só compra depois de se cadastrar: ajusta cadastro quando necessário
    cadastro = np.minimum(cli["data_cadastro"].values, (dt.dt.normalize() - pd.Timedelta(days=1)).values)
    idade_conta_dias = ((dt.dt.normalize().values - cadastro) / np.timedelta64(1, "D")).astype(int)

    canal = rng.choice(["app", "site", "loja física"], size=n, p=[0.46, 0.30, 0.24])
    fisica = canal == "loja física"
    meio = np.where(fisica,
                    rng.choice(["cartão de crédito", "cartão de débito", "Pix"], size=n, p=[0.55, 0.25, 0.20]),
                    rng.choice(["cartão de crédito", "Pix", "boleto"], size=n, p=[0.62, 0.30, 0.08]))
    cat_principal = rng.choice(list(CATEGORIAS), size=n, p=[0.16, 0.10, 0.12, 0.22, 0.16, 0.10, 0.14])
    preco_med_cat = produtos.groupby("categoria")["preco_lista"].median().reindex(cat_principal).values
    seg_mult = cli["segmento"].map({"novo": 0.85, "recorrente": 1.0, "vip": 1.9}).values
    qtd_itens = 1 + rng.poisson(0.9, n)
    valor = (preco_med_cat * seg_mult * rng.lognormal(0, 0.55, n) * (0.7 + 0.3 * qtd_itens)).round(2)
    parcelas = np.where(meio == "cartão de crédito",
                        np.clip(np.round(valor / rng.uniform(150, 600, n)), 1, 12), 1).astype(int)

    hora = dt.dt.hour.values
    madrugada = (hora >= 0) & (hora <= 5)
    dispositivo_novo = np.where(fisica, False, rng.random(n) < 0.07)
    end_alterado_24h = np.where(fisica, False, rng.random(n) < 0.03)
    distancia_km = np.where(fisica, rng.exponential(6, n), rng.exponential(18, n)).round(1)
    tentativas_pagto = 1 + rng.poisson(0.15, n)

    # confundidor: VIP recebe mais cupom E gasta mais (cupom não causa gasto) -> aula 3
    p_cupom = cli["segmento"].map({"novo": 0.18, "recorrente": 0.25, "vip": 0.55}).values
    usou_cupom = rng.random(n) < p_cupom

    # --- probabilidade de fraude (log-odds) ---
    pos_drift = (dt >= pd.Timestamp("2026-07-01")).values      # novo golpe a partir de jul/2026
    z = (-5.9
         + 1.60 * dispositivo_novo
         + 1.20 * madrugada
         + 0.75 * np.log1p(valor / 500)
         + 1.10 * (idade_conta_dias < 30)
         + 0.50 * (tentativas_pagto > 1)
         + 0.35 * (distancia_km > 60)
         + 0.55 * ((meio == "cartão de crédito") & (parcelas >= 10))
         + 0.40 * end_alterado_24h
         + np.where(cat_principal == "Eletrônicos", 0.65, 0.0)
         - 2.2 * fisica)
    # concept drift (jul/2026): golpe via Pix em conta antiga invadida, de dia, no aparelho já conhecido.
    # Os sinais antigos perdem força e surgem sinais novos -> o modelo treinado antes envelhece.
    z = z + pos_drift * (2.6 * end_alterado_24h + 1.6 * (meio == "Pix") * ~fisica
                         - 1.3 * dispositivo_novo - 1.0 * madrugada
                         - 0.9 * (idade_conta_dias < 30) - 0.5 * np.log1p(valor / 500))
    p = 1 / (1 + np.exp(-z))
    fraude = rng.random(n) < p

    # maturação do rótulo: contestação chega de 3 a 120 dias depois
    atraso = np.clip(rng.gamma(2.2, 14, n), 3, 120).astype(int)
    data_confirmacao = dt.dt.normalize() + pd.to_timedelta(atraso, "D")
    confirmada = fraude & (data_confirmacao.values <= DATA_CORTE.normalize())
    data_confirmacao = data_confirmacao.where(confirmada, pd.NaT)

    # ARMADILHA (target leakage): status do pedido só é conhecido DEPOIS do fato
    status = np.where(confirmada, "cancelado_chargeback",
                      np.where(rng.random(n) < 0.04, "cancelado_cliente", "entregue"))
    status = np.where(confirmada & (rng.random(n) < 0.08), "entregue", status)  # ruído realista

    df = pd.DataFrame({
        "id_transacao": [f"T{i:07d}" for i in range(1, n + 1)],
        "data_hora": dt.values,
        "id_cliente": cli["id_cliente"].values,
        "canal": canal,
        "meio_pagamento": meio,
        "parcelas": parcelas,
        "categoria_principal": cat_principal,
        "qtd_itens": qtd_itens,
        "valor_total": valor,
        "usou_cupom": usou_cupom,
        "idade_conta_dias": idade_conta_dias,
        "dispositivo_novo": dispositivo_novo,
        "endereco_alterado_24h": end_alterado_24h,
        "distancia_entrega_km": distancia_km,
        "tentativas_pagamento": tentativas_pagto,
        "uf_entrega": np.where(fisica, cli["uf"].values,
                               np.where(rng.random(n) < 0.9, cli["uf"].values, rng.choice(UFS, n, p=PESO_UF))),
        "status_pedido": status,                       # LEAKAGE
        "fraude_confirmada": confirmada.astype(int),    # TARGET (Case A)
        "data_confirmacao_fraude": data_confirmacao.values,
    })

    # ARMADILHA (vazamento temporal): contagem de contestações do cliente usando TODO o histórico
    contest = df.groupby("id_cliente")["fraude_confirmada"].transform("sum")
    df["qtd_contestacoes_cliente"] = contest.astype(int)        # LEAKAGE temporal

    # ARMADILHA (qualidade): ~0,3% de linhas duplicadas por reenvio de integração
    dup = df.sample(frac=0.003, random_state=int(rng.integers(0, 1_000_000)))
    df = pd.concat([df, dup]).sort_values(["data_hora", "id_transacao"]).reset_index(drop=True)
    return df


# ----------------------------------------------------------------------------
# Case B — vendas semanais (previsão de vendas)
# ----------------------------------------------------------------------------
def gerar_vendas_semanais(rng, produtos, lojas):
    semanas = pd.date_range(DATA_INICIO, DATA_CORTE, freq="W-MON")
    semanas = semanas[semanas + pd.Timedelta(days=6) <= DATA_CORTE]
    P, L, S = len(produtos), len(lojas), len(semanas)
    grade = pd.MultiIndex.from_product([semanas, lojas["id_loja"], produtos["id_produto"]],
                                       names=["semana_inicio", "id_loja", "id_produto"]).to_frame(index=False)
    grade = grade.merge(produtos[["id_produto", "categoria", "preco_lista", "data_lancamento"]], on="id_produto")
    grade = grade.merge(lojas[["id_loja", "uf", "tipo"]], on="id_loja")
    grade = grade[grade["semana_inicio"] >= grade["data_lancamento"]].reset_index(drop=True)
    n = len(grade)

    sem = grade["semana_inicio"]
    t = ((sem - DATA_INICIO).dt.days / 365.25).values
    mes = sem.dt.month.values
    woy = sem.dt.isocalendar().week.values.astype(int)

    base_prod = dict(zip(produtos["id_produto"], rng.lognormal(1.6, 0.7, P)))
    base = grade["id_produto"].map(base_prod).values
    preco_rel = (grade["preco_lista"].values / 500) ** -0.35      # item caro vende menos
    porte_loja = grade["id_loja"].map({"L01": 1.4, "L02": 0.9, "L03": 1.1, "L04": 0.9,
                                       "L05": 0.8, "L06": 0.7, "ECOM": 2.6}).values
    sazonal_natal = grade["categoria"].map({c: v[4] for c, v in CATEGORIAS.items()}).values
    sazonal = 1 + 0.12 * np.sin(2 * np.pi * (woy - 10) / 52)
    sazonal = sazonal * np.where(mes == 12, sazonal_natal, 1.0)

    feriado = np.isin(sem.dt.strftime("%m-%d").values, []) | False
    black_friday = np.zeros(n, dtype=bool)
    for ano in (2024, 2025, 2026):
        bf_dia = pd.Timestamp(f"{ano}-11-01") + pd.offsets.Week(weekday=4) * 4
        black_friday |= (sem.values == (bf_dia - pd.Timedelta(days=bf_dia.weekday())).to_datetime64())
    semana_feriado = np.isin(woy, [1, 8, 15, 17, 36, 41, 44, 46, 47, 52])
    promocao = (rng.random(n) < 0.12) | black_friday
    desconto = np.where(promocao, rng.choice([0.10, 0.15, 0.20, 0.30], size=n, p=[0.35, 0.3, 0.25, 0.10]), 0.0)
    desconto = np.where(black_friday, np.maximum(desconto, 0.25), desconto)
    preco_praticado = (grade["preco_lista"].values * (1 - desconto)).round(2)

    tendencia = 1 + 0.10 * t + np.where(grade["categoria"].values == "Casa Inteligente", 0.22 * t, 0.0)
    efeito_promo = (1 - desconto) ** -2.0                          # elasticidade
    # drift: concorrente entra em ago/2026 e derruba Eletrônicos nas lojas de SP
    choque = np.where((sem.values >= np.datetime64("2026-08-03")) & (grade["categoria"].values == "Eletrônicos")
                      & (grade["uf"].values == "SP"), 0.72, 1.0)
    lam = (base * preco_rel * porte_loja * sazonal * tendencia * efeito_promo
           * np.where(black_friday, 2.4, 1.0) * np.where(semana_feriado, 1.08, 1.0) * choque)
    qtd = rng.negative_binomial(5, 5 / (5 + lam))

    df = pd.DataFrame({
        "semana_inicio": sem.dt.date.values,
        "id_loja": grade["id_loja"].values,
        "id_produto": grade["id_produto"].values,
        "categoria": grade["categoria"].values,
        "preco_lista": grade["preco_lista"].values,
        "preco_praticado": preco_praticado,
        "em_promocao": promocao,
        "desconto_pct": desconto,
        "semana_black_friday": black_friday,
        "semana_com_feriado": semana_feriado,
        "qtd_vendida": qtd,                                             # TARGET 1 (Case B)
        "receita_bruta": (qtd * preco_praticado).round(2),              # TARGET 2 / LEAKAGE p/ qtd
    })
    return df


# ----------------------------------------------------------------------------
def gerar_base(seed=42):
    """Gera todas as tabelas. seed=42 é a versão oficial da disciplina."""
    rng = _rng(seed)
    lojas = gerar_lojas()
    produtos = gerar_produtos(rng)
    clientes = gerar_clientes(rng)
    transacoes = gerar_transacoes(rng, clientes, produtos)
    vendas = gerar_vendas_semanais(rng, produtos, lojas)
    return {"lojas": lojas, "produtos": produtos, "clientes": clientes,
            "transacoes": transacoes, "vendas_semanais": vendas}


def impressao_digital(df):
    """Hash do conteúdo de uma tabela: se mudar um único valor, o hash muda."""
    return hashlib.sha256(pd.util.hash_pandas_object(df, index=True).values.tobytes()).hexdigest()[:16]


def resumo(tabelas):
    linhas = []
    for nome, df in tabelas.items():
        linhas.append({"tabela": nome, "linhas": len(df), "colunas": df.shape[1],
                       "hash": impressao_digital(df)})
    return pd.DataFrame(linhas)


if __name__ == "__main__":
    import os, sys
    destino = sys.argv[1] if len(sys.argv) > 1 else "loja_aurora_v1"
    os.makedirs(destino, exist_ok=True)
    tabs = gerar_base(42)
    for nome, df in tabs.items():
        df.to_csv(os.path.join(destino, f"{nome}.csv"), index=False)
    print(resumo(tabs).to_string(index=False))

In [0]:
import numpy as np, pandas as pd
import loja_aurora_gerador as aurora

def br(x, casas=0):
    # formata números no padrão brasileiro: 1.234,56
    return f"{x:,.{casas}f}".replace(",", "X").replace(".", ",").replace("X", ".")

tabelas = aurora.gerar_base(seed=42)
display(aurora.resumo(tabelas))

✅ Confira o hash de `transacoes` com o da Aula 1 (`793d63c17ff5e699` no ambiente de geração). Se for diferente, anote as versões de numpy e pandas: o **ambiente** também é versão (Aula 3).

## 2. As premissas do grupo (edite aqui)
Todo número de negócio depende de premissas. O primeiro passo da governança é **escrevê-las** em um único lugar, com nome e valor, para que qualquer pessoa possa questionar e refazer a conta.

In [0]:
PREMISSAS = {
    "versao_canvas":            "1.0.0",
    "custo_revisao_manual_rs":  8.00,   # custo de um analista revisar uma compra (R$)
    "eficacia_revisao":         0.90,   # % das fraudes revisadas que o analista consegue barrar
    "desistencia_cliente_bom":  0.10,   # % dos bons clientes que desistem quando a compra cai em revisão
    "margem_media":             0.30,   # margem bruta média da venda online
    "tarifa_chargeback_rs":     30.00,  # tarifa cobrada pelo adquirente em cada chargeback (R$)
    "limite_regra_atual_rs":    2000,   # regra de hoje: compra acima deste valor vai para revisão
    "custo_estoque_semana":     0.02,   # Case B: custo de carregar 1 unidade parada por semana (% do custo)
}
PREMISSAS

## 3. Bloco 1 e Bloco 2 · Quem usa e qual o tamanho da dor
Usamos 2025 inteiro: um ano fechado, com o rótulo de fraude já **maduro** (guarde esta palavra para daqui a pouco).

In [0]:
tx = tabelas["transacoes"].drop_duplicates("id_transacao")          # remove os reenvios da integração
online = tx[tx["canal"].isin(["app", "site"])].copy()
ano = online[online["data_hora"].dt.year == 2025]

compras_mes      = len(ano) / 12
receita_online   = ano["valor_total"].sum()
fraudes          = ano[ano["fraude_confirmada"] == 1]
perda_fraude     = fraudes["valor_total"].sum() + len(fraudes) * PREMISSAS["tarifa_chargeback_rs"]

print(f"Compras online por mês (média 2025): {br(compras_mes)}")
print(f"Receita online 2025:                 R$ {br(receita_online/1e6, 1)} milhões")
print(f"Fraudes confirmadas em 2025:         {br(len(fraudes))} ({br(len(fraudes)/len(ano)*100, 2)}% das compras online)")
print(f"Perda com fraude + tarifas:          R$ {br(perda_fraude/1e6, 2)} milhões ({br(perda_fraude/receita_online*100, 2)}% da receita online)")

## 4. Blocos 5 e 9 · Quanto vale decidir melhor, e quanto custa
Comparamos três políticas de decisão no checkout, com a **mesma capacidade** do time de revisão:

| Política | Como decide |
|---|---|
| Aprovar tudo | Nenhuma revisão. É a referência de "não fazer nada". |
| Regra atual | Compra acima de R$ 2 mil vai para revisão manual (a regra da Aula 1). |
| Score simples | Um modelo bem simples, treinado em 2024, ordena as compras de 2025 por risco. Revisamos as mais arriscadas, no mesmo número da regra. |

Para cada política, calculamos em reais: fraude evitada, custo da revisão e venda perdida por atrito com bons clientes. É a **matriz de confusão em reais**, o "custo do erro" do canvas.

In [0]:
from sklearn.linear_model import LogisticRegression

def pistas(df):
    return pd.DataFrame({
        "log_valor":        np.log1p(df["valor_total"]),
        "dispositivo_novo": df["dispositivo_novo"].astype(int),
        "madrugada":        (df["data_hora"].dt.hour <= 5).astype(int),
        "conta_nova":       (df["idade_conta_dias"] < 30).astype(int),
        "endereco_24h":     df["endereco_alterado_24h"].astype(int),
        "varias_tentativas":(df["tentativas_pagamento"] > 1).astype(int),
        "parcelas_10_mais": (df["parcelas"] >= 10).astype(int),
        "eletronicos":      (df["categoria_principal"] == "Eletrônicos").astype(int),
    })

treino_2024 = online[online["data_hora"].dt.year == 2024]
score = LogisticRegression(max_iter=1000).fit(pistas(treino_2024), treino_2024["fraude_confirmada"])
ano = ano.assign(risco=score.predict_proba(pistas(ano))[:, 1])

def avaliar(nome, revisar):
    p = PREMISSAS
    f, bom = ano["fraude_confirmada"] == 1, ano["fraude_confirmada"] == 0
    barradas = ano[revisar & f]
    evitada  = p["eficacia_revisao"] * (barradas["valor_total"].sum() + len(barradas) * p["tarifa_chargeback_rs"])
    custo    = revisar.sum() * p["custo_revisao_manual_rs"]
    perdida  = p["desistencia_cliente_bom"] * p["margem_media"] * ano.loc[revisar & bom, "valor_total"].sum()
    return {"política": nome, "revisões": int(revisar.sum()), "fraudes pegas": int((revisar & f).sum()),
            "% fraudes pegas": round((revisar & f).sum() / f.sum() * 100, 1),
            "fraude evitada (R$ mil)": round(evitada / 1e3), "custo revisão (R$ mil)": round(custo / 1e3),
            "venda perdida (R$ mil)": round(perdida / 1e3), "resultado líquido (R$ mil)": round((evitada - custo - perdida) / 1e3)}

regra = ano["valor_total"] > PREMISSAS["limite_regra_atual_rs"]
capacidade = int(regra.sum())
top_risco = ano["risco"].rank(ascending=False, method="first") <= capacidade

resultado = pd.DataFrame([
    avaliar("Aprovar tudo", pd.Series(False, index=ano.index)),
    avaliar("Regra atual (> R$ 2 mil)", regra),
    avaliar("Score simples (mesma capacidade)", top_risco),
])
resultado

💡 **Leitura de negócio:** com o mesmo time de revisão, ordenar por risco pega mais fraudes e incomoda menos bons clientes do que olhar só o valor da compra.
O resultado líquido vai para o bloco 5 (valor gerado) e os custos para o bloco 9. **Mude uma premissa** (por exemplo, `desistencia_cliente_bom = 0.25`) e rode de novo: a decisão continua de pé?

### Case B · Quanto custa errar a previsão de vendas
Para os grupos de previsão de vendas: usamos uma previsão ingênua (média das 4 semanas até duas semanas antes) e medimos, em 2025, o custo de prever **de menos** (ruptura: margem perdida) e **de mais** (estoque parado).

In [0]:
v = tabelas["vendas_semanais"].merge(tabelas["produtos"][["id_produto", "custo_unitario"]], on="id_produto")
v["semana_inicio"] = pd.to_datetime(v["semana_inicio"])
v = v.sort_values(["id_loja", "id_produto", "semana_inicio"])
v["previsao_ingenua"] = (v.groupby(["id_loja", "id_produto"])["qtd_vendida"]
                          .transform(lambda s: s.shift(2).rolling(4).mean()))
b = v[(v["semana_inicio"].dt.year == 2025) & v["previsao_ingenua"].notna()].copy()

falta = (b["qtd_vendida"] - b["previsao_ingenua"]).clip(lower=0)
sobra = (b["previsao_ingenua"] - b["qtd_vendida"]).clip(lower=0)
custo_ruptura = (falta * (b["preco_praticado"] - b["custo_unitario"])).sum()
custo_estoque = (sobra * b["custo_unitario"] * PREMISSAS["custo_estoque_semana"]).sum()
wmape = (b["qtd_vendida"] - b["previsao_ingenua"]).abs().sum() / b["qtd_vendida"].sum()

print(f"Erro da previsão ingênua (WMAPE): {br(wmape*100, 1)}%")
print(f"Margem perdida por ruptura (2025): R$ {br(custo_ruptura/1e6, 2)} milhões")
print(f"Custo de estoque parado (2025):    R$ {br(custo_estoque/1e6, 2)} milhões")

## 5. O canvas como código
Em vez de um post-it que se perde, o canvas vira um **dicionário Python**: cada bloco tem texto e, quando existe, um número calculado acima.
Assim ele pode ser **regerado, comparado e versionado** como o resto do projeto.

In [0]:
r = resultado.set_index("política")
liquido = r.loc["Score simples (mesma capacidade)", "resultado líquido (R$ mil)"] - r.loc["Regra atual (> R$ 2 mil)", "resultado líquido (R$ mil)"]

CANVAS = {
    "produto": "Guardião do Checkout · Loja Aurora",
    "case": "A · Prevenção a fraude",
    "versao": PREMISSAS["versao_canvas"],
    "blocos": {
        "1 Segmentos":      f"Usa: time de prevenção a fraude. Paga: diretoria financeira. Beneficia: cliente do app e do site ({br(compras_mes)} compras/mês).",
        "2 Proposta de valor": f"Decidir em menos de 1 s aprovar, revisar ou recusar cada compra online. Dor atual: R$ {br(perda_fraude/1e6, 2)} mi/ano em fraude e tarifas ({br(perda_fraude/receita_online*100, 2)}% da receita online).",
        "3 Canais":         "Score no checkout via API e fila de revisão no painel do analista.",
        "4 Relacionamento": "Bom cliente aprovado na hora. Compra em revisão recebe contato pelo WhatsApp em até 15 min.",
        "5 Fontes de receita": f"Valor gerado: + R$ {br(liquido)} mil/ano frente à regra atual, com a mesma equipe (premissas v{PREMISSAS['versao_canvas']}).",
        "6 Recursos-chave": "34 meses de transações e contestações, sinais de dispositivo e endereço, time de analistas.",
        "7 Atividades-chave": "Definir a target, versionar dados e modelo, revisar casos, monitorar drift e retreinar.",
        "8 Parcerias-chave": "Adquirente e bandeiras (chargebacks), bureau antifraude, nuvem de dados (Databricks).",
        "9 Estrutura de custos": f"Revisão manual R$ {br(r.loc['Score simples (mesma capacidade)', 'custo revisão (R$ mil)'])} mil/ano, venda perdida por atrito R$ {br(r.loc['Score simples (mesma capacidade)', 'venda perdida (R$ mil)'])} mil/ano, time de dados e nuvem.",
    },
    "premissas": PREMISSAS,
}
for bloco, texto in CANVAS["blocos"].items():
    print(f"{bloco:22s} {texto}")

In [0]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
import textwrap

VERDE, NEON, CREME, CINZA = "#023327", "#00E482", "#E8E1DC", "#5F6B66"

def desenhar_canvas(canvas, arquivo="canvas_produto_ia.png"):
    fig, ax = plt.subplots(figsize=(16, 9), dpi=110)
    ax.set_xlim(0, 100); ax.set_ylim(0, 100); ax.axis("off")
    fig.patch.set_facecolor("white")
    ax.text(1, 97, "AI BUSINESS MODEL CANVAS", color=VERDE, fontsize=11, weight="bold")
    ax.text(1, 92.5, canvas["produto"], color=VERDE, fontsize=20, weight="bold")
    ax.text(99, 94, f"Case {canvas['case']}  ·  versão {canvas['versao']}", color=CINZA, fontsize=10, ha="right")
    col = [1, 20.8, 40.6, 60.4, 80.2]; w = 18.8
    caixas = {  # bloco: (x, y, largura, altura)
        "8 Parcerias-chave": (col[0], 30, w, 59), "7 Atividades-chave": (col[1], 60.5, w, 28.5),
        "6 Recursos-chave": (col[1], 30, w, 28.5), "2 Proposta de valor": (col[2], 30, w, 59),
        "4 Relacionamento": (col[3], 60.5, w, 28.5), "3 Canais": (col[3], 30, w, 28.5),
        "1 Segmentos": (col[4], 30, w, 59), "9 Estrutura de custos": (1, 2, 48.5, 26),
        "5 Fontes de receita": (50.5, 2, 48.5, 26),
    }
    for bloco, (x, y, bw, bh) in caixas.items():
        destaque = bloco.startswith("2")
        ax.add_patch(FancyBboxPatch((x, y), bw, bh, boxstyle="round,pad=0,rounding_size=1.2",
                                    fc=CREME if destaque else "white", ec=NEON if destaque else VERDE, lw=2 if destaque else 1))
        num, nome = bloco.split(" ", 1)
        ax.add_patch(FancyBboxPatch((x + 1, y + bh - 5.2), 3.6, 4.2, boxstyle="round,pad=0,rounding_size=0.8", fc=VERDE, ec="none"))
        ax.text(x + 2.8, y + bh - 3.1, num, color=NEON, fontsize=11, weight="bold", ha="center", va="center")
        ax.text(x + 5.6, y + bh - 3.1, nome, color=VERDE, fontsize=11, weight="bold", va="center")
        largura = int(bw * 1.75)
        texto = "\n".join(textwrap.wrap(canvas["blocos"][bloco], largura)).replace("$", r"\$")
        ax.text(x + 1.2, y + bh - 7.5, texto, color="#1D2B26", fontsize=9, va="top", linespacing=1.35)
    ax.text(99, 0, "Adaptado do Business Model Canvas (Osterwalder e Pigneur). Base sintética Loja Aurora v1.", color=CINZA, fontsize=7, ha="right")
    fig.savefig(arquivo, bbox_inches="tight", facecolor="white")
    plt.show()

desenhar_canvas(CANVAS)

## 6. Versionar o canvas
O canvas é o **contrato do grupo com o negócio**. Se uma premissa muda, o número muda, e alguém vai perguntar "o que mudou desde a última reunião?".
Por isso salvamos o canvas em dois formatos e calculamos uma **impressão digital** (hash), como fizemos com as tabelas na Aula 1:
- `canvas_produto_ia.json` · para a máquina (comparar versões)
- `docs/canvas_produto_ia.md` · para as pessoas (vai para o GitLab)

In [0]:
import json, hashlib, os, datetime

def salvar_canvas(canvas):
    os.makedirs("docs", exist_ok=True)
    conteudo = json.dumps(canvas, ensure_ascii=False, indent=2, sort_keys=True)
    impressao = hashlib.sha256(conteudo.encode("utf-8")).hexdigest()[:16]
    with open("canvas_produto_ia.json", "w", encoding="utf-8") as f:
        f.write(conteudo)
    linhas = [f"# {canvas['produto']}", "",
              f"**Case:** {canvas['case']} · **Versão:** {canvas['versao']} · **Hash:** `{impressao}` · **Gerado em:** {datetime.date.today():%d/%m/%Y}", "",
              "| Bloco | Conteúdo |", "|---|---|"]
    linhas += [f"| {b} | {t} |" for b, t in canvas["blocos"].items()]
    linhas += ["", "## Premissas", "", "| Premissa | Valor |", "|---|---|"]
    linhas += [f"| `{k}` | {v} |" for k, v in canvas["premissas"].items()]
    with open("docs/canvas_produto_ia.md", "w", encoding="utf-8") as f:
        f.write("\n".join(linhas))
    return impressao

hash_v1 = salvar_canvas(CANVAS)
print("Canvas v1 salvo · impressão digital:", hash_v1)

### Simulando a reunião seguinte
O financeiro informa que a revisão manual custa **R$ 12**, não R$ 8. O grupo atualiza a premissa, sobe a versão para **1.1.0** e refaz a conta.

In [0]:
import copy, difflib

CANVAS_V1 = copy.deepcopy(CANVAS)
PREMISSAS["custo_revisao_manual_rs"] = 12.00
PREMISSAS["versao_canvas"] = "1.1.0"

resultado = pd.DataFrame([avaliar("Regra atual (> R$ 2 mil)", regra), avaliar("Score simples (mesma capacidade)", top_risco)]).set_index("política")
liquido = resultado.loc["Score simples (mesma capacidade)", "resultado líquido (R$ mil)"] - resultado.loc["Regra atual (> R$ 2 mil)", "resultado líquido (R$ mil)"]
CANVAS["versao"] = "1.1.0"
CANVAS["premissas"] = dict(PREMISSAS)
CANVAS["blocos"]["5 Fontes de receita"] = f"Valor gerado: + R$ {br(liquido)} mil/ano frente à regra atual, com a mesma equipe (premissas v1.1.0)."
CANVAS["blocos"]["9 Estrutura de custos"] = (f"Revisão manual R$ {br(resultado.loc['Score simples (mesma capacidade)', 'custo revisão (R$ mil)'])} mil/ano, "
                                            f"venda perdida por atrito R$ {br(resultado.loc['Score simples (mesma capacidade)', 'venda perdida (R$ mil)'])} mil/ano, time de dados e nuvem.")
hash_v11 = salvar_canvas(CANVAS)
print("v1.0.0:", hash_v1, "  →  v1.1.0:", hash_v11)
print()
antes  = json.dumps(CANVAS_V1, ensure_ascii=False, indent=2, sort_keys=True).splitlines()
depois = json.dumps(CANVAS,    ensure_ascii=False, indent=2, sort_keys=True).splitlines()
for linha in difflib.unified_diff(antes, depois, "canvas v1.0.0", "canvas v1.1.0", lineterm="", n=0):
    print(linha)

💡 Repare: o custo da revisão subiu, mas o **ganho do score frente à regra não mudou**, porque as duas políticas revisam o mesmo número de compras. Sem a conta versionada, essa conclusão viraria discussão de opinião.

O **diff** mostra exatamente o que mudou entre as versões: é o que o GitLab vai mostrar quando o grupo fizer o *commit*.
Sugestão de mensagem de commit: `canvas v1.1.0: custo de revisão atualizado para R$ 12 (fonte: financeiro, 16/11)`.

**Para o grupo (5 min):** troque os textos do dicionário `CANVAS` pelos do canvas de vocês, rode `desenhar_canvas` e `salvar_canvas` e suba `docs/canvas_produto_ia.md` no repositório.

In [0]:
# Baixar os arquivos para subir no GitLab (no Colab)
try:
    from google.colab import files
    for arquivo in ["docs/canvas_produto_ia.md", "canvas_produto_ia.json", "canvas_produto_ia.png"]:
        files.download(arquivo)
except ImportError:
    print("Fora do Colab: os arquivos estão na pasta atual.")

## 7. O elo com a aula de hoje: e se o número da dor estiver errado?
O bloco 2 usou **2025**, um ano fechado. E se a diretoria pedir a perda com fraude **de 2026 até agora**?

In [0]:
tx26 = online[online["data_hora"] >= "2026-01-01"].copy()
tx26["mes"] = tx26["data_hora"].dt.to_period("M").dt.to_timestamp()
mensal = tx26.groupby("mes")["fraude_confirmada"].mean() * 100

fig, ax = plt.subplots(figsize=(11, 3.8), dpi=110)
cores = [VERDE if m < pd.Timestamp("2026-09-01") else NEON for m in mensal.index]
MESES = ["jan", "fev", "mar", "abr", "mai", "jun", "jul", "ago", "set", "out", "nov", "dez"]
rotulos = [f"{MESES[m.month - 1]}/{m:%y}" for m in mensal.index]
ax.bar(rotulos, mensal.values, color=cores)
for i, val in enumerate(mensal.values):
    ax.text(i, val + 0.03, br(val, 2) + "%", ha="center", fontsize=9, color=VERDE)
ax.set_title("Taxa de fraude observada hoje nas compras online, por mês da compra", loc="left", color=VERDE, weight="bold")
ax.spines[["top", "right", "left"]].set_visible(False); ax.set_yticks([])
ax.text(len(mensal) - 1.5, mensal.max() * 0.62, "set e out:\nrótulo ainda\nnão amadureceu", color=VERDE, fontsize=9, ha="center")
plt.show()

❓ **Pergunta para a turma:** em outubro a taxa cai para cerca de um terço. Se esse número entrar no bloco 2, a diretoria vai concluir que o problema está sumindo.

A resposta está em uma pergunta que ainda não fizemos com cuidado: **o que exatamente é uma "fraude" e quando sabemos que ela aconteceu?**
Esse é o tema da aula de hoje: **construir a target sem se enganar**. Vamos para o Databricks.